# BTL Học sâu – Đề tài 8 (chạy trên Google Colab)
**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**. Chạy các ô theo thứ tự từ trên xuống.
Mỗi bước có ghi rõ *đầu ra mong đợi*. Nếu một bước lỗi, dừng lại, copy thông báo lỗi để hỏi AI Agent, đừng chạy bước tiếp theo.
Toàn bộ kết quả được lưu trên Google Drive nên mất runtime vẫn không mất dữ liệu.

## Bước 0. Cấu hình (chỉ sửa ô này)

In [ ]:
REPO_URL   = "https://github.com/RoyalAnh/word2vec_and_downstream_evaluation"   # repo GitHub chứa thư mục dự án (nếu để trống sẽ dùng Drive)
REPO_DIR   = ""                    # mã nguồn nằm ngay gốc repo
DRIVE_CODE = "/content/drive/MyDrive/btl_w2v_vi"            # chỉ dùng khi REPO_URL để trống
N_ARTICLES = 20000                 # số bài Wikipedia DÀI (>=1500 ký tự). Tách từ chậm thì giảm xuống 10000
VSFC_DIR   = ""                    # để trống = tải UIT-VSFC qua Hugging Face; nếu lỗi, điền đường dẫn thư mục UIT-VSFC trên Drive
NEURAL_KV  = "w2v_sg_w3_d300"      # mô hình w2v dùng cho CNN/LSTM (có thể đổi sau khi xem Bảng 1)
ABL_CFG    = "sg_w3_d300"          # cấu hình w2v cho ablation ngữ liệu

## Bước 1. Gắn Drive, lấy mã, cài thư viện

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

if REPO_URL.strip():
    if not os.path.exists('/content/repo'):
        subprocess.run(['git','clone',REPO_URL,'/content/repo'],check=True)
    else:
        subprocess.run(['git','-C','/content/repo','pull'],check=True)   # lấy mã mới nhất vừa push
    CODE = os.path.join('/content/repo', REPO_DIR) if REPO_DIR else '/content/repo'
else:
    CODE = DRIVE_CODE
assert os.path.exists(os.path.join(CODE,'src','prepare_data.py')), f"Không thấy src/prepare_data.py trong {CODE}"

WORK = '/content/drive/MyDrive/btl_work'          # nơi lưu dữ liệu, mô hình, kết quả (giữ lại sau khi tắt Colab)
os.makedirs(WORK, exist_ok=True)
os.environ['BTL_WORK'] = WORK
os.environ['PYTHONHASHSEED'] = '0'
os.chdir(CODE)
print('CODE =', CODE, '| WORK =', WORK)

In [ ]:
def need(*rel):
    """Dừng ngay nếu bước trước chưa tạo ra file cần thiết (tránh một loạt Traceback dây chuyền)."""
    miss = [r for r in rel if not os.path.exists(os.path.join(WORK, r))]
    assert not miss, f"THIẾU {miss} -> bước trước đó đã lỗi. Sửa lỗi ở bước đó rồi chạy lại, ĐỪNG chạy tiếp."
    print("OK, đã có:", rel)

In [ ]:
!pip install -q gensim pyvi datasets transformers scikit-learn matplotlib "pandas==2.2.3" tqdm
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())
# ĐẦU RA MONG ĐỢI: in ra 'GPU: True'. Nếu gặp lỗi numpy/gensim 'binary incompatibility' -> Runtime > Restart session rồi chạy lại từ Bước 1.

## Bước 2. (Tuỳ chọn, ~10 giây) Chạy thử bằng dữ liệu GIẢ
Chỉ để kiểm tra môi trường ổn. Dữ liệu giả ghi ra `/tmp/smoke`, **không lẫn** vào kết quả thật và **không dùng để nộp**.

In [ ]:
!BTL_WORK=/tmp/smoke python src/prepare_data.py --smoke --n_jobs 1 | tail -3
!BTL_WORK=/tmp/smoke python src/train_w2v.py --epochs 1 | tail -2
# ĐẦU RA MONG ĐỢI: in thống kê và '[done] w2v_...'. Không có Traceback là ổn.

## Bước 2b. (CHỈ khi chạy lại từ đầu với ngữ liệu mới) Cất kết quả cũ sang thư mục lưu trữ
Chuyển (không xoá) `data/`, `models/`, `results/` của lần chạy trước sang `archive_<giờ>/` trên Drive để không lẫn vào lần chạy mới.
`train_w2v.py` **bỏ qua** mô hình đã có sẵn nên nếu không cất đi, nó sẽ dùng lại mô hình cũ (huấn luyện trên ngữ liệu cũ).
Lần chạy đầu tiên (Drive trống) thì bỏ qua ô này.

In [ ]:
import shutil, time
arch = f"{WORK}/archive_{time.strftime('%Y%m%d_%H%M')}"
moved = []
for d in ("data", "models", "results"):
    if os.path.exists(f"{WORK}/{d}"):
        os.makedirs(arch, exist_ok=True)
        shutil.move(f"{WORK}/{d}", f"{arch}/{d}")
        moved.append(d)
print("Đã chuyển:", moved, "->", arch if moved else "(không có gì để chuyển)")
# ĐẦU RA MONG ĐỢI: 'Đã chuyển: [data, models, results] -> .../archive_...'. Sau đó btl_work chỉ còn các thư mục archive_*.

## Bước 3. Dữ liệu THẬT: Wikipedia tiếng Việt + UIT-VSFC (tải tự động, có thể mất 10–40 phút)
**Đầu ra mong đợi** trong `btl_work/`: `data/wiki_tokens.txt`, `data/vsfc_{train,dev,test}.jsonl`, `results/data_stats.json`, `results/figures/zipf_wiki.png`, `zipf_vsfc.png`, `sentence_length_wiki.png`.
Kiểm tra: dòng `[wiki] duyệt ... bài, giữ ... bài dài`; `[VSFC]` ~ 11k/1.5k/3k; **ngữ liệu vài triệu token và vocab_min_count > 10.000**; cuối log có mục `[KIỂM TRA NGỮ LIỆU]` với *từ mẫu có trong từ vựng ≥ 15/31*.
Nếu thấy `[CẢNH BÁO] ngữ liệu còn nhỏ` hoặc từ mẫu < 15: **dừng**, đừng chạy Bước 4 (gửi log Bước 3 cho AI Agent).

In [ ]:
vs = f'--vsfc_dir "{VSFC_DIR}"' if VSFC_DIR else ''
!python src/prepare_data.py --n_articles {N_ARTICLES} {vs}
need("data/wiki_tokens.txt", "data/vsfc_train.jsonl", "data/vsfc_dev.jsonl", "data/vsfc_test.jsonl", "results/data_stats.json")
# ĐẦU RA MONG ĐỢI: dòng "[VSFC] train/dev/test = {...}" (~11k/1.5k/3k), rồi "OK, đã có: ..."

## Bước 4. Huấn luyện embedding (word2vec ×8, SVD ×4, GloVe ×4)
**Đầu ra mong đợi:** `models/` có 8 file `w2v_*.kv` + 4 `svd_*.kv` + 4 `glove_*.kv`. Mỗi dòng `[done] ...` báo vocab và thời gian. (~30–90 phút, tuỳ ngữ liệu.)

In [ ]:
need("data/wiki_tokens.txt")
!python src/train_w2v.py --workers 1
!python src/cooc_svd_glove.py
need(*[f"models/{n}.kv" for n in ["w2v_sg_w3_d300","w2v_cbow_w8_d100"]])

## Bước 5. Đánh giá nội tại
**Đầu ra mong đợi:** mỗi mô hình một dòng `[tên] analogy_acc=... covered=x/294 sample_found=...`; `results/tables/neighbors_*.md`; `results/figures/tsne_*.png`, `pca_*.png`.
**⚠ DỪNG & KIỂM TRA:** nếu `covered` rất thấp (<100/294) hoặc có `[CẢNH BÁO] ... <15 từ mẫu`, danh sách từ trong `src/eval_intrinsic.py` không khớp từ vựng. Mở `neighbors_*.md` và `data/wiki_tokens.txt`, sửa danh sách rồi chạy lại ô này (hoặc nhờ AI Agent sửa).

In [ ]:
need("data/wiki_tokens.txt")
!python src/eval_intrinsic.py

## Bước 6. Đánh giá ngoại tại – cảm xúc (sentiment)
**Đầu ra mong đợi:** các dòng `[linear] ...`, `[mean_mlp] <mô hình>: macro-F1 = x ± y` (16 dòng), `[neural] ... seed=...` (18 dòng), `[phobert frozen]`, `[phobert finetune] seed=...` (3 dòng). Kết quả ghi vào `results/experiment_log.csv`. (~40–90 phút.)

In [ ]:
need("data/vsfc_train.jsonl")
!python src/eval_extrinsic.py --task sentiment --stage all --neural_kv {NEURAL_KV}

In [ ]:
need("data/vsfc_train.jsonl")
!python src/eval_phobert.py --task sentiment --mode both

## Bước 7. Ngoại tại – chủ đề (topic) (tuỳ chọn, nên làm nếu còn thời gian)

In [ ]:
need("data/vsfc_train.jsonl")
!python src/eval_extrinsic.py --task topic --stage all --neural_kv {NEURAL_KV}

In [ ]:
need("data/vsfc_train.jsonl")
!python src/eval_phobert.py --task topic --mode both

## Bước 8. Ablation ngữ liệu + thiên kiến
**Đầu ra mong đợi:** 5 dòng `[wiki_10] ... [wiki_100+vsfc] ...`; `results/figures/corpus_ablation.png`; `[mô hình] WEAT: {...}` và `results/tables/bias_projection_*.md`.

In [ ]:
need("data/wiki_tokens.txt", "data/vsfc_train.jsonl")
!python src/corpus_ablation.py --cfg {ABL_CFG} --task sentiment
!python src/bias_analysis.py

## Bước 9. Tổng hợp bảng + biểu đồ
**Đầu ra mong đợi:** `results/summary_sentiment.md` (Bảng 1–6), `results/tables/table*_sentiment.md`, `results/figures/representations_compare_sentiment.png`.

In [ ]:
need("results/experiment_log.csv")
!python src/aggregate.py --task sentiment
!python src/aggregate.py --task topic || echo "bỏ qua topic nếu chưa chạy Bước 7" 

## Bước 10. Đóng gói kết quả để tải về
Tạo `results_pack.zip` (nhật ký, bảng, hình, thống kê, nhãn+văn bản VSFC đã xử lý; **không** gồm mô hình nặng).
Tải về rồi **giải nén vào thư mục `notebooks/output/`** của repo.

In [ ]:
need("results/experiment_log.csv")
!cd {WORK} && zip -qr /content/results_pack.zip results data/vsfc_*.jsonl && ls -lh /content/results_pack.zip
!cp /content/results_pack.zip {WORK}/results_pack.zip
from google.colab import files
files.download('/content/results_pack.zip')

## Xem nhanh nhật ký thí nghiệm

In [ ]:
import pandas as pd
df = pd.read_csv(f'{WORK}/results/experiment_log.csv')
print(len(df), 'lượt chạy'); df.groupby('experiment').size()